# **Multiclass Classification: One-vs-All and All-Pairs**
### Team: Yes Free Dinner
### GitHub Repository: https://github.com/SimaRaha/multiclass-classification-from-scratch
### Team Members: George Chudley, Sima Raha, Yuan Ying, Kunhao Yue

## **Overview of One-vs-All and All-Pairs Multiclass Classification**

Many classification problems involve more than two possible labels—such as recognizing a handwritten digit (ten classes), identifying a plant species, or routing a support ticket to one of several teams. However, foundational learning algorithms like logistic regression, the perceptron, and support vector machines are naturally binary: they separate examples into exactly two groups. A standard and effective way to handle multiple classes is through reduction: breaking the multiclass problem into several binary sub-problems, training one binary classifier for each, and combining their outputs into a single prediction (Shalev-Shwartz and Ben-David, 2014, Section 17.1).

In this project we implement the two reductions covered in Lecture 5, using binary logistic regression as the underlying binary classifier:

- **One-vs-all (OvA)**, also called one-vs-rest, trains $K$ classifiers. Classifier $k$ learns to separate class $k$ from all other classes combined. At prediction time, the class whose classifier is most confident wins.

- **All-pairs (AP)**, also called one-vs-one, sets up head-to-head matchups between every possible pair of classes, $\binom{K}{2} = K(K-1)/2$ classifiers in total. Classifier $(i, j)$ is trained only on examples from classes $i$ and $j$. At prediction time, every classifier casts a vote, and the class with the most votes wins.

We chose logistic regression because it produces a real-valued confidence score, not only a hard label. Both reductions need such a score: OvA uses it to pick the most confident class, and AP uses it to break ties between classes with the same number of votes.

**Advantages:**

- **Reuses any binary learner.** No new optimisation algorithm is needed; the multiclass model is built entirely from a binary classifier.
- **Simple and parallel.** Every binary classifier is trained independently, so training can be split across processors.
- **Strong in practice.** Rifkin and Klautau (2004) show that OvA with a well-tuned binary classifier is as accurate as more complicated multiclass methods. AP classifiers each solve a smaller, often easier problem, because they only need to separate two classes (Hastie and Tibshirani, 1998).


**Disadvantages:**

- **Classifiers are trained in isolation.** None of the binary classifiers knows it is part of a multiclass system, so their confidence scores are not guaranteed to be on comparable scales (Shalev-Shwartz and Ben-David, 2014, Section 17.1).
- **Class imbalance in OvA.** Each OvA classifier sees one positive class against $K-1$ negative classes. With $K$ balanced classes, only about $1/K$ of its training examples are positive, which is about 10% for the ten digit classes.
- **Many models for AP.** AP trains one classifier per pair of classes, $\binom{K}{2} = \frac{K(K-1)}{2}$ in total, so the number of classifiers grows quadratically with $K$: ten classes need $10 \times 9 / 2 = 45$ classifiers, and 100 classes would need 4,950. In contrast, OvA needs only $K$.
- **Ambiguous regions.** Combining binary decisions can leave regions of feature space where several classes tie, so an explicit tie-breaking rule is required (Bishop, 2006, Section 4.1.2).

### Representation

Each binary classifier is a logistic regression model (Shalev-Shwartz and Ben-David, 2014, Section 9.3). An example is described by $d$ features; for the Digits data these are the $d = 64$ pixel intensities of an $8 \times 8$ image (Alpaydin and Kaynak, 1998). We append a constant feature equal to $1$ to every example, so that the bias is learned as an ordinary weight. The input is therefore a vector $\mathbf{x} \in \mathbb{R}^{d+1}$, and the model has a weight vector $\mathbf{w} \in \mathbb{R}^{d+1}$.

The model first combines the features into a single score, the weighted sum $z = \mathbf{w}^T \mathbf{x}$. It then passes this score through the **sigmoid function**, which squashes it into a probability between $0$ and $1$:

$$
h_{\mathbf{w}}(\mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x}) = \frac{1}{1 + e^{-\mathbf{w}^T \mathbf{x}}} = P(y = 1 \mid \mathbf{x}).
$$

The binary prediction is $1$ if $h_{\mathbf{w}}(\mathbf{x}) \ge 0.5$, which happens exactly when $\mathbf{w}^T \mathbf{x} \ge 0$, and $0$ otherwise. The decision boundary $\mathbf{w}^T \mathbf{x} = 0$ is a hyperplane, and the further an example lies from it, the more confident the prediction (Shalev-Shwartz and Ben-David, 2014, Section 9.3). A single model can only separate two classes, so to predict one of $K$ classes we combine several binary models using one of two reductions (Shalev-Shwartz and Ben-David, 2014, Section 17.1).

**One-vs-All (OvA).** We train $K$ binary models, one per class. Model $k$ has weight vector $\mathbf{w}_k$ and is trained with the examples of class $k$ labelled $1$ and all other examples labelled $0$. A new example is assigned to the class whose model is most confident:

$$
\hat{y} = \arg\max_{k \in \{1, \dots, K\}} h_{\mathbf{w}_k}(\mathbf{x}) = \arg\max_{k \in \{1, \dots, K\}} \mathbf{w}_k^T \mathbf{x}.
$$

The two forms give the same class because the sigmoid is increasing. In our code we compare the scores $\mathbf{w}_k^T \mathbf{x}$ directly, because very large scores can all round to a probability of exactly $1.0$ and create artificial ties. For Digits ($K = 10$), OvA uses 10 models.

**All-Pairs (AP).** We train one binary model for every pair of classes $i < j$, which gives $\frac{K(K-1)}{2}$ models (45 for Digits). Model $(i, j)$ has weight vector $\mathbf{w}_{ij}$ and is trained only on the examples of classes $i$ and $j$, with class $j$ labelled $1$ and class $i$ labelled $0$. We use this labelling convention from scikit-learn (Pedregosa et al., 2011) so that our results can be compared with it directly. To classify a new example, each model votes for class $j$ if $h_{\mathbf{w}_{ij}}(\mathbf{x}) \ge 0.5$, and for class $i$ otherwise. The predicted class is the one with the most votes:

$$
\hat{y} = \arg\max_{c \in \{1, \dots, K\}} V_c(\mathbf{x}), \qquad V_c(\mathbf{x}) = \text{number of pairwise models that vote for class } c.
$$

**Tie-breaking in All-Pairs.** Because votes are whole numbers, several classes can receive the same number of votes. Following scikit-learn, we break ties with the models' scores. Each class $c$ collects a confidence equal to the scores that favour it minus the scores that favour its opponents:

$$
C_c(\mathbf{x}) = \sum_{i < c} \mathbf{w}_{ic}^T \mathbf{x} \; - \sum_{j > c} \mathbf{w}_{cj}^T \mathbf{x}.
$$

This confidence is squashed into the interval $\left(-\tfrac13, \tfrac13\right)$ and added to the votes:

$$
\hat{y} = \arg\max_{c} \left( V_c(\mathbf{x}) + \frac{C_c(\mathbf{x})}{3\,\big(|C_c(\mathbf{x})| + 1\big)} \right).
$$

The added terms of any two classes differ by less than $\tfrac23$, so they can never overturn a difference of one or more votes; they only decide between classes that are tied (scikit-learn developers, n.d.).

### Loss

Each binary model is trained by minimizing the **regularized binary cross-entropy loss** (log loss) with an $L_2$ (Tikhonov) penalty. Given $N$ training examples $\{(\mathbf{x}^{(n)}, y^{(n)})\}_{n=1}^N$, where $y^{(n)} \in \{0, 1\}$ is the label for that model's binary sub-problem, the objective is

$$
J(\mathbf{w}) = -\frac{1}{N} \sum_{n=1}^{N} \left[ y^{(n)} \log h_{\mathbf{w}}(\mathbf{x}^{(n)}) + \left(1 - y^{(n)}\right) \log\left(1 - h_{\mathbf{w}}(\mathbf{x}^{(n)})\right) \right] + \lambda \lVert \mathbf{w} \rVert_2^2 .
$$

The first term is the log loss (Shalev-Shwartz and Ben-David, 2014, Section 9.3). For each example, only one of its two parts is active, and it equals $-\log$ of the probability the model gave to the true label. The loss is close to $0$ when the model is confident and correct, and it grows without bound as the model becomes confident and wrong, so confident mistakes are penalized heavily. The log loss is also convex in $\mathbf{w}$, so gradient-based training cannot get stuck in a poor local minimum.

The second term, $\lambda \lVert \mathbf{w} \rVert_2^2$, discourages large weights and helps prevent overfitting (Shalev-Shwartz and Ben-David, 2014, Section 13.1). Without it, logistic regression can push its weights towards infinity when the classes are linearly separable (Bishop, 2006, Section 4.3.2). The hyperparameter $\lambda \ge 0$ controls how strongly large weights are penalized. As in our homework, the penalty is applied to all weights, including the bias.

The final multiclass predictions are evaluated with **accuracy**, the fraction of correctly classified examples, which is one minus the 0-1 loss. The 0-1 loss is not used for training, because it is flat almost everywhere and provides no useful gradient.

### Optimizer

Unlike linear regression, logistic regression has no closed-form solution for the weights that minimize $J(\mathbf{w})$, so each binary model is trained iteratively with **mini-batch stochastic gradient descent (SGD)** (Shalev-Shwartz and Ben-David, 2014, Chapter 14). Instead of computing the gradient on all $N$ examples, SGD estimates it on a small batch. The estimate is noisy but correct on average, and each update is much cheaper to compute.

The weights start at $\mathbf{w} = \mathbf{0}$. In each epoch, the training data is shuffled and split into mini-batches of size $B$. For each mini-batch $\mathcal{B}$, we compute the gradient of the regularized loss,

$$
\nabla_{\mathbf{w}} J_{\mathcal{B}}(\mathbf{w}) = \frac{1}{\lvert \mathcal{B} \rvert} \sum_{n \in \mathcal{B}} \left( h_{\mathbf{w}}(\mathbf{x}^{(n)}) - y^{(n)} \right) \mathbf{x}^{(n)} + 2\lambda \mathbf{w},
$$

and move the weights a small step in the opposite direction, with learning rate $\eta$:

$$
\mathbf{w} \leftarrow \mathbf{w} - \eta \, \nabla_{\mathbf{w}} J_{\mathcal{B}}(\mathbf{w}).
$$

We train for a fixed number of epochs. Each model shuffles with a fixed random seed, so training is deterministic and every run produces the same weights, which we need in order to reproduce scikit-learn's results exactly. We use $\eta = 0.1$, $B = 15$, $\lambda = 0.001$ and 100 epochs. Because the Digits pixel values range from 0 to 16, we divide them by 16 so that all features lie in $[0, 1]$, which helps SGD converge.

The two reductions do not change the optimizer. They only decide which examples and labels each binary model is trained on, and how the trained models are combined at prediction time.

The pseudo-code below follows the stochastic gradient descent, one-vs-all and all-pairs algorithms of Shalev-Shwartz and Ben-David (2014, Chapter 14 and Section 17.1), adapted to labels $y \in \{0, 1\}$, an $L_2$ penalty, and scikit-learn's labelling and tie-breaking conventions so that our results can be compared with scikit-learn directly.

#### Pseudo-Code: Training One Binary Model

```text
ALGORITHM: Binary_Train(X, y, lr, lambda, epochs, batch_size, seed)
  Append a column of ones to X                       # bias feature
  Initialize weights w = zeros(number of columns of X)     # d + 1, including the bias
  rng = random number generator with the given seed
  FOR epoch = 1 to epochs DO
      Shuffle the rows of (X, y) using rng
      FOR each mini-batch B of size batch_size DO
          p = sigmoid(B.X @ w)
          gradient = (1 / |B|) * B.X.T @ (p - B.y) + 2 * lambda * w
          w = w - lr * gradient
      END FOR
  END FOR
  RETURN w
```

#### Pseudo-Code: One-vs-All

```text
ALGORITHM: OneVsAll_Train(X, Y, lr, lambda, epochs, batch_size, seed)
  classes = sorted unique labels in Y               # K classes
  models = []
  FOR each class k in classes DO
      y_bin = 1 where Y == k, 0 otherwise
      Append Binary_Train(X, y_bin, lr, lambda, epochs, batch_size, seed) to models
  END FOR
  RETURN classes, models

ALGORITHM: OneVsAll_Predict(X, classes, models)
  S = score matrix of shape (num_samples, K)
  FOR k = 1 to K DO
      S[:, k] = scores of models[k] on X             # w_k^T x
  END FOR
  RETURN classes[argmax(S, axis=1)]
```

#### Pseudo-Code: All-Pairs

```text
ALGORITHM: AllPairs_Train(X, Y, lr, lambda, epochs, batch_size, seed)
  classes = sorted unique labels in Y               # K classes
  models = []
  FOR each pair (i, j) of class indices with i < j DO
      X_kept, Y_kept = the rows of (X, Y) where Y == classes[i] or Y == classes[j]
      y_bin = 1 where Y_kept == classes[j], 0 where Y_kept == classes[i]
      Append ((i, j), Binary_Train(X_kept, y_bin, lr, lambda, epochs, batch_size, seed)) to models
  END FOR
  RETURN classes, models

ALGORITHM: AllPairs_Predict(X, classes, models)
  votes = zeros(num_samples, K)
  conf  = zeros(num_samples, K)
  FOR each ((i, j), model) in models DO
      s = scores of model on X                       # w_ij^T x
      votes[:, j] += 1 where s >= 0
      votes[:, i] += 1 where s < 0
      conf[:, j] += s
      conf[:, i] -= s
  END FOR
  total = votes + conf / (3 * (abs(conf) + 1))
  RETURN classes[argmax(total, axis=1)]
```

## **Model**

In [3]:
import numpy as np

def sigmoid_function(x):
    return 1.0 / (1.0 + np.exp(-x))

class RegularizedLogisticRegression(object):
    '''
    Implement regularized logistic regression for binary classification.
    The weight vector w should be learned by minimizing the regularized loss
    L(h, (x,y)) = log(1 + exp(-y <w, x>)) + lambda |w|_2^2. In other words, the objective
    function that we are trying to minimize is the log loss for binary logistic regression 
    plus Tikhonov regularization with a coefficient of lambda.
    '''
    def __init__(self, batch_size = 15):
        self.learningRate = 0.00001 # Feel free to play around with this if you'd like, though this value will do
        self.num_epochs = 10000 # Feel free to play around with this if you'd like, though this value will do
        self.batch_size = batch_size # Feel free to play around with this if you'd like, though this value will do
        self.weights = None
        self.lmbda = 1

    def train(self, X, Y):
        '''
        Train the model, using batch stochastic gradient descent
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the corresponding labels for each example
        @return:
            None
        '''
        self.weights = np.zeros((1, X.shape[1]))
        for _ in range(self.num_epochs):
            indices = np.random.permutation(X.shape[0])
            X_shuffle = X[indices]
            Y_shuffle = Y[indices]
            for start in range(0, X.shape[0], self.batch_size):
                xbatch = X_shuffle[start : start + self.batch_size]
                ybatch = Y_shuffle[start : start + self.batch_size]
                Lw = np.zeros((1, X.shape[1]))
                for k in range(len(xbatch)):
                    h_k = sigmoid_function(self.weights[0] @ xbatch[k])
                    Lw += (h_k - ybatch[k]) * xbatch[k]
                Lw /= len(xbatch)  
                Lw += 2 * self.lmbda * self.weights
                self.weights -= self.learningRate * Lw

    def predict(self, X):
        '''
        Compute predictions based on the learned parameters and examples X
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            A 1D Numpy array with one element for each row in X containing the predicted class.
        '''
        result = sigmoid_function(X @ self.weights.T).flatten()
        return np.where(result >= 0.5, 1, 0)

    def accuracy(self,X, Y):
        '''
        Output the accuracy of the trained model on a given testing dataset X and labels Y.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the corresponding labels for each example
        @return:
            a float number indicating accuracy (between 0 and 1)
        '''
        count = 0
        result = self.predict(X)
        for i in range(X.shape[0]):
            if result[i] == Y[i]:
                count += 1
        return count / X.shape[0]

    def predict_proba(self, X):
        '''
        Compute the predicted probabilities of class 1 for each example.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            a 1D Numpy array containing the predicted probability of class 1 for each example
        '''
        return sigmoid_function(X @ self.weights.T).flatten()

    def loss(self, X, Y):
        '''
        Compute the mean binary logistic loss with L2 regularization:
        L(w) = -1/m * sum(y_i * log(h(x_i)) + (1 - y_i) * log(1 - h(x_i))) + lambda * ||w||_2^2
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the binary labels (0 or 1)
        @return:
            a float representing the mean binary logistic loss plus the L2 regularization term
        '''
        h = sigmoid_function(X @ self.weights.T).flatten()
        eps = 1e-15
        h = np.clip(h, eps, 1 - eps)
        bce_loss = -np.mean(Y * np.log(h) + (1 - Y) * np.log(1 - h))
        reg_loss = self.lmbda * np.sum(self.weights ** 2)
        return float(bce_loss + reg_loss)


class OneVsAllClassifier(object):
    '''
    Implement one-vs-all classification using regularized logistic regression.
    For each class, train a separate binary classifier to distinguish that class from all other classes. 
    During prediction, select the class with the highest predicted probability.
    '''

    def __init__(self, batch_size=15):
        '''
        Initialize the training parameters and model storage.
        @params:
            batch_size: the number of examples used in each mini-batch
        @return:
            None
        '''
        self.learningRate = 0.00001
        self.num_epochs = 10000
        self.batch_size = batch_size
        self.lmbda = 1
        self.classes = None
        self.models = []

    def train(self, X, Y):
        '''
        Train one binary logistic regression model for each class.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the original class labels
        @return:
            None
        '''
        self.classes = np.unique(Y)
        self.models = []

        for cls in self.classes:
            binary_Y = (Y == cls).astype(int)
            model = RegularizedLogisticRegression(self.batch_size)
            model.learningRate = self.learningRate
            model.num_epochs = self.num_epochs
            model.lmbda = self.lmbda
            model.train(X, binary_Y)
            self.models.append(model)

    def predict(self, X):
        '''
        Predict the class labels using the trained binary classifiers.
        The class with the highest predicted probability is selected.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            a 1D Numpy array containing the predicted class label for each example
        '''
        scores = np.column_stack([model.predict_proba(X) for model in self.models])
        return self.classes[np.argmax(scores, axis=1)]

    def loss(self, X, Y):
        '''
        Compute the average regularized logistic loss across all
        binary classifiers.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the original class labels
        @return:
            a float representing the mean loss across all binary classifiers
        '''
        losses = []
        for cls, model in zip(self.classes, self.models):
            binary_Y = (Y == cls).astype(int)
            losses.append(model.loss(X, binary_Y))
        return float(np.mean(losses))

    def accuracy(self, X, Y):
        '''
        Compute the classification accuracy on a given dataset.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the original class labels
        @return:
            a float representing the fraction of correctly classified examples (between 0 and 1)
        '''
        return float(np.mean(self.predict(X) == Y))


In [1]:
import numpy as np
import pandas as pd
from itertools import combinations

class AllPairsMulticlassLogisticRegression(object):
    '''
    Implement all-pairs classification using regularized logistic regression.
    For each pair of classes, train a separate binary classifier to distinguish the two classes.
    During prediction, select the class with the majority counts across all classifiers.
    '''
    def __init__(self):
        '''
        Initialize the training parameters and model storage.
        @params:
            None
        @return:
            None
        '''
        self.learning_rate = 0.00001
        self.num_epochs = 10000
        self.weights = None
        self.lamb = 1
        self.classes = None
        
    def train(self, X, y):
        '''
        Train one binary logistic regression model for each pair of classes.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            y: a 2D Numpy array where each row contains a class label for the corresponding row in X
        @return:
            A 1D Numpy array with one element for each row in X containing the predicted class.
        '''
        n, d = X.shape
        self.classes = np.unique(y)
        self.pairs = list(combinations(self.classes, 2))
        k = len(self.classes)
        P_count = len(self.pairs)

        Y = np.column_stack([(y == a) for a, b in self.pairs]).astype(float)
        M = np.column_stack([(y == a) | (y == b) for a, b in self.pairs]).astype(float)
        m = M.sum(axis=0)

        self.weights = np.zeros((d,P_count))

        for _ in range(self.num_epochs):
            P = self.sigmoid_function(X @ self.weights)
            error = (P - Y) * M
            grad_weights = (X.T @ error) / m + (self.lamb / m) * self.weights
            self.weights -= self.learning_rate * grad_weights

        return self

    def predict_probs(self, X):
        '''
        Predict the probability of each class for each classifier pair.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            Y: a 2D Numpy array where each row contains a predicted probability for a pair of classes
        '''
        return self.sigmoid_function(X @ self.weights)

    def predict(self, X):
        '''
        Predict the class labels using the trained binary classifiers.
        The class predicted most frequently and with the highest probability by classifiers is selected.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            a 1D Numpy array containing the predicted class label for each example
        '''
        probs = self.predict_probs(X)
        k = len(self.classes)
        idx = {c: i for i, c in enumerate(self.classes)}
        votes = np.zeros((X.shape[0], k))
        conf = np.zeros((X.shape[0], k))
        for p, (a, b) in enumerate(self.pairs):
            votes[:, idx[a]] += probs[:, p] >= 0.5
            votes[:, idx[b]] += probs[:, p] < 0.5
            conf[:, idx[a]] += probs[:, p]
            conf[:, idx[b]] += 1 - probs[:, p]
        return self.classes[np.argmax(votes + conf / k, axis=1)]

    def loss(self, X, Y):
        '''
        Compute the average regularized logistic loss across all
        binary classifiers.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the original class labels
        @return:
            a float representing the mean loss across all binary classifiers
        '''
        probs = np.clip(self.predict_probs(X), 1e-12, 1 - 1e-12)
        losses = []
        for p, (a, b) in enumerate(self.pairs):
            mask = (Y == a) | (Y == b)
            yp, pp = (Y[mask] == a).astype(float), probs[mask, p]
            nll = -np.mean(yp * np.log(pp) + (1 - yp) * np.log(1 - pp))
            losses.append(nll + self.lamb / (2 * mask.sum()) * np.sum(self.weights[:, p] ** 2))
        return float(np.mean(losses))

    def accuracy(self, X, Y):
        '''
        Compute the classification accuracy on a given dataset.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the original class labels
        @return:
            a float representing the fraction of correctly classified examples (between 0 and 1)
        '''
        return float(np.mean(self.predict(X) == Y))

## **Check Model**

## **References**

Alpaydin, E. and Kaynak, C. (1998) *Optical Recognition of Handwritten Digits* [Dataset]. UCI Machine Learning Repository. Available at: https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits (Accessed: 16 November 2026).

Bishop, C.M. (2006) *Pattern Recognition and Machine Learning*. New York: Springer. Available at: https://www.microsoft.com/en-us/research/publication/pattern-recognition-machine-learning/ (Accessed: 16 November 2026).

Hastie, T. and Tibshirani, R. (1998) 'Classification by pairwise coupling', *The Annals of Statistics*, 26(2), pp. 451–471. doi: 10.1214/aos/1028144844.

Pedregosa, F. et al. (2011) 'Scikit-learn: Machine learning in Python', *Journal of Machine Learning Research*, 12, pp. 2825–2830. Available at: https://jmlr.org/papers/v12/pedregosa11a.html (Accessed: 16 November 2026).

Rifkin, R. and Klautau, A. (2004) 'In defense of one-vs-all classification', *Journal of Machine Learning Research*, 5, pp. 101–141. Available at: https://www.jmlr.org/papers/v5/rifkin04a.html (Accessed: 16 November 2026).

scikit-learn developers (n.d.) *Multiclass and multioutput algorithms*. Available at: https://scikit-learn.org/stable/modules/multiclass.html (Accessed: 16 November 2026).

Shalev-Shwartz, S. and Ben-David, S. (2014) *Understanding Machine Learning: From Theory to Algorithms*. Cambridge: Cambridge University Press. doi: 10.1017/CBO9781107298019.
